In [3]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd


class CabinPassengerViewer:
    def __init__(self, db_file):
        self.db_file = db_file
        self.conn = sqlite3.connect(self.db_file)

        self.root = tk.Tk()
        self.root.title("Cabin Passenger Viewer")
        self.root.geometry("750x450")

        self.create_widgets()
        self.load_cabins()

        self.root.protocol("WM_DELETE_WINDOW", self.close_app)
        self.root.mainloop()

    def create_widgets(self):
        
        title_label = tk.Label(
            self.root,
            text="Choose a Cabin to View Its Passengers",
            font=("Arial", 14, "bold")
        )
        title_label.pack(pady=10)

        # frame for controls
        top_frame = tk.Frame(self.root)
        top_frame.pack(pady=10)

        cabin_label = tk.Label(top_frame, text="Cabin:")
        cabin_label.grid(row=0, column=0, padx=5, pady=5)

        self.cabin_combo = ttk.Combobox(top_frame, width=25, state="readonly")
        self.cabin_combo.grid(row=0, column=1, padx=5, pady=5)

        show_button = tk.Button(
            top_frame,
            text="Show Passengers",
            command=self.show_passengers
        )
        show_button.grid(row=0, column=2, padx=10, pady=5)

        # cabin details label
        self.details_label = tk.Label(
            self.root,
            text="Cabin details will appear here",
            font=("Arial", 11),
            fg="blue"
        )
        self.details_label.pack(pady=5)

        # treeview for passengers
        columns = ("passenger_id", "personal_name", "family_name", "age", "sex")
        self.tree = ttk.Treeview(self.root, columns=columns, show="headings", height=12)

        self.tree.heading("passenger_id", text="Passenger ID")
        self.tree.heading("personal_name", text="First Name")
        self.tree.heading("family_name", text="Family Name")
        self.tree.heading("age", text="Age")
        self.tree.heading("sex", text="Sex")

        self.tree.column("passenger_id", width=90, anchor="center")
        self.tree.column("personal_name", width=140, anchor="center")
        self.tree.column("family_name", width=140, anchor="center")
        self.tree.column("age", width=80, anchor="center")
        self.tree.column("sex", width=80, anchor="center")

        self.tree.pack(fill="both", expand=True, padx=15, pady=10)

        # scrollbar
        scrollbar = ttk.Scrollbar(self.root, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        scrollbar.place(x=715, y=115, height=255)

    def load_cabins(self):
        """
        Load cabins into the dropdown.
        You can change the display text here if needed.
        """
        query = """
        SELECT cabin_id, class, ship_level
        FROM cabin_inventory
        ORDER BY cabin_id
        """
        df = pd.read_sql(query, self.conn)

        # Store display text mapped to cabin_id
        self.cabin_map = {}
        display_values = []

        for _, row in df.iterrows():
            display_text = f"Cabin {row['cabin_id']} - {row['class']} - {row['ship_level']}"
            display_values.append(display_text)
            self.cabin_map[display_text] = row["cabin_id"]

        self.cabin_combo["values"] = display_values

        if display_values:
            self.cabin_combo.current(0)

    def show_passengers(self):
        selected_cabin = self.cabin_combo.get()

        if not selected_cabin:
            messagebox.showwarning("No Selection", "Please choose a cabin first.")
            return

        cabin_id = self.cabin_map[selected_cabin]

        # Get cabin details
        cabin_query = """
        SELECT cabin_id, class, no_of_beds, price, ship_level
        FROM cabin_inventory
        WHERE cabin_id = ?
        """
        cabin_df = pd.read_sql(cabin_query, self.conn, params=(cabin_id,))

        if cabin_df.empty:
            messagebox.showerror("Error", "Cabin not found.")
            return

        cabin = cabin_df.iloc[0]
        self.details_label.config(
            text=(
                f"Cabin {cabin['cabin_id']} | "
                f"Class: {cabin['class']} | "
                f"Deck: {cabin['ship_level']} | "
                f"Beds: {cabin['no_of_beds']} | "
                f"Price: {cabin['price']}"
            )
        )

        # Get passengers in selected cabin
        passenger_query = """
        SELECT passenger_id, personal_name, family_name, age, sex
        FROM passenger
        WHERE cabin_id = ?
        ORDER BY family_name, personal_name
        """
        passenger_df = pd.read_sql(passenger_query, self.conn, params=(cabin_id,))
        #print(passenger_df)
        # Clear old rows
        for item in self.tree.get_children():
            self.tree.delete(item)

        # Insert new rows
        
        if passenger_df.empty:
            messagebox.showinfo("No Passengers", "There are no passengers assigned to this cabin.")
        else:
            for _, row in passenger_df.iterrows():
                self.tree.insert(
                    "",
                    "end",
                    values=(
                        row["passenger_id"],
                        row["personal_name"],
                        row["family_name"],
                        row["age"],
                        row["sex"]
                    )
                )
        
    def close_app(self):
        self.conn.close()
        self.root.destroy()


# Run the program
if __name__ == "__main__":
    CabinPassengerViewer("example.db")

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\jo861\AppData\Local\anaconda3_v3\Lib\site-packages\pandas\core\indexes\base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'personal_name'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\jo861\AppData\Local\anaconda3_v3\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\jo861\AppData\Local\Temp\